# S23DR 2026 — RoofWireframeNet Training

**Runtime**: GPU (Runtime → Change runtime type → T4 GPU)  
**Expected time**: ~14 min/epoch on T4, ~7 hrs for 100 epochs on A100

Steps:
1. Install dependencies
2. Clone repo and set up package
3. Verify GPU + quick smoke test
4. Run full training
5. Download checkpoint

In [ ]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 2. Install dependencies ──────────────────────────────────────────────────
!pip install -q datasets huggingface_hub scipy

In [ ]:
# ── 3. Clone repo ────────────────────────────────────────────────────────────
import os
if not os.path.exists('3d_building_construction'):
    !git clone https://github.com/12turtleships/3d_building_construction.git
%cd 3d_building_construction
!git log --oneline -3

In [ ]:
# ── 4. Verify imports ────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')
from s23dr.model import RoofWireframeNet, WireframeLoss, N_EDGE_CLASSES
from s23dr.data  import S23DRDataset, collate_fn

model = RoofWireframeNet(n_queries=64)
total = sum(p.numel() for p in model.parameters())
print(f'Model: {total:,} parameters')
print('OK')

In [ ]:
# ── 5. Smoke test: forward pass on GPU ──────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using:', device)

model = RoofWireframeNet(n_queries=64).to(device)
loss_fn = WireframeLoss()

B, N = 4, 1024
xyz  = torch.randn(B, N, 3).to(device)
vf   = torch.rand(B, N).to(device)
nv   = torch.randint(0, 8, (B, N)).float().to(device)
msk  = torch.ones(B, N).to(device)
cid  = torch.randint(0, 10, (B, N)).to(device)
gv   = [torch.randn(31, 3) for _ in range(B)]
ge   = [torch.randint(0, 31, (32, 2)) for _ in range(B)]
gc   = [torch.randint(0, 10, (32,)) for _ in range(B)]

out    = model(xyz, vf, nv, msk, cid)
losses = loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'], gv, ge, gc)
print('pred_pos:', tuple(out['pred_pos'].shape))
print('loss:    ', losses['loss'].item())
print('Smoke test passed!')

In [ ]:
# ── 6. Benchmark step time on GPU ────────────────────────────────────────────
import time
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Warm-up
for _ in range(3):
    out = model(xyz, vf, nv, msk, cid)
    loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'], gv, ge, gc)['loss'].backward()
    opt.step(); opt.zero_grad()
if device == 'cuda': torch.cuda.synchronize()

times = []
for _ in range(10):
    t0 = time.time()
    out = model(xyz, vf, nv, msk, cid)
    loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'], gv, ge, gc)['loss'].backward()
    opt.step(); opt.zero_grad()
    if device == 'cuda': torch.cuda.synchronize()
    times.append(time.time() - t0)

avg = sum(times) / len(times)
steps_per_epoch = 15892 // B
secs_per_epoch  = avg * steps_per_epoch
print(f'Step time:   {avg*1000:.0f} ms  (batch={B}, n_points=1024)')
print(f'Steps/epoch: {steps_per_epoch}')
print(f'Time/epoch:  {secs_per_epoch/60:.1f} min')
print(f'100 epochs:  {secs_per_epoch*100/3600:.1f} hrs')

In [ ]:
# ── 7. Load dataset (downloads from HF, cached after first run) ─────────────
print('Loading train split (~15 k samples)…')
train_ds = S23DRDataset(split='train',      n_points=1024)
val_ds   = S23DRDataset(split='validation', n_points=1024)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

In [ ]:
# ── 8. Full training ─────────────────────────────────────────────────────────
# Adjust EPOCHS / BATCH_SIZE to fit your runtime budget.
# T4  (free Colab):  BATCH_SIZE=8,  EPOCHS=50  → ~7 hrs
# A100 (Colab Pro+): BATCH_SIZE=16, EPOCHS=100 → ~4 hrs

import time
from pathlib import Path
from torch.utils.data import DataLoader
import torch.optim as optim

EPOCHS     = 100
BATCH_SIZE = 8
LR         = 1e-3
N_QUERIES  = 64
CKPT_DIR   = Path('outputs/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn, drop_last=True,
                          pin_memory=(device=='cuda'))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate_fn,
                          pin_memory=(device=='cuda'))

model     = RoofWireframeNet(n_queries=N_QUERIES).to(device)
loss_fn   = WireframeLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'Training {sum(p.numel() for p in model.parameters()):,} params on {device}')
print(f'{EPOCHS} epochs × {len(train_loader)} steps/epoch')

best_val = float('inf')
LOG_EVERY = 200

for epoch in range(1, EPOCHS + 1):
    # ---- train ----
    model.train()
    epoch_loss = 0.0
    t0 = time.time()
    for step, batch in enumerate(train_loader):
        xyz  = batch['xyz'].to(device)
        vf   = batch['vote_frac'].to(device)
        nv   = batch['n_views'].to(device)
        msk  = batch['mask'].to(device)
        cid  = batch['class_id'].to(device)
        out  = model(xyz, vf, nv, msk, cid)
        loss = loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'],
                       batch['gt_verts'], batch['gt_edges'], batch['gt_classes'])['loss']
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        if (step + 1) % LOG_EVERY == 0:
            print(f'  ep{epoch} step{step+1}/{len(train_loader)}  '
                  f'loss={loss.item():.4f}  ({time.time()-t0:.0f}s)')
    # ---- val ----
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch['xyz'].to(device), batch['vote_frac'].to(device),
                        batch['n_views'].to(device), batch['mask'].to(device),
                        batch['class_id'].to(device))
            val_loss += loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'],
                                batch['gt_verts'], batch['gt_edges'],
                                batch['gt_classes'])['loss'].item()
    val_loss /= len(val_loader)
    train_loss = epoch_loss / len(train_loader)
    scheduler.step()
    elapsed = time.time() - t0
    print(f'Epoch {epoch:3d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  ({elapsed:.0f}s)')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'val_loss': val_loss}, CKPT_DIR / 'best.pt')
        print(f'  ✓ checkpoint saved (val={val_loss:.4f})')

print(f'Done. Best val loss: {best_val:.4f}')

In [ ]:
# ── 9. Download checkpoint ───────────────────────────────────────────────────
from google.colab import files
files.download('outputs/checkpoints/best.pt')

In [ ]:
# ── 10. (Optional) Save to Google Drive instead ─────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('outputs/checkpoints/best.pt',
#             '/content/drive/MyDrive/s23dr_best.pt')
# print('Saved to Drive')